# Setup

In [1]:
!pip install -q datasets transformers evaluate accelerate
!pip install -q "ray[tune]" scipy sklearn torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 474.3/474.3 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 13.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 24.4.1 requires pyarrow<15.0.0a0,>=14.0.1, but you have pyarrow 17.0.0 which is incompatible.
ibis-framework 8.0.0 requires pyarrow<16,>=2, but you have pyarrow 17.0.0 which is incompatible.
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  

In [2]:
!pip -q install lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.0/811.0 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.2/869.2 kB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 815.2/815.2 kB 52.2 MB/s eta 0:00:00


In [3]:
import os, sys
import pickle
from time import gmtime, strftime
from tqdm.notebook import tqdm  # Progress bar

import matplotlib as plt
import seaborn as sns
import numpy as np
import pandas as pd
import scipy as sp

import torch
import torch.nn as nn
from torch import Tensor
from torch.utils.data import Dataset, DataLoader
from lightning.pytorch.utilities import CombinedLoader
from sklearn.metrics import f1_score, classification_report

from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification, AutoModel, AutoConfig
from transformers import DebertaV2Config, DebertaV2Model, DebertaV2PreTrainedModel
from transformers import DataCollatorWithPadding
from transformers import TrainingArguments, Trainer
from transformers import get_scheduler


In [4]:
'''Set-up GPU for use'''

gpu_avail = torch.cuda.is_available()
print(f"Is the GPU available? {gpu_avail}")

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("Device", device)

# GPU operations have a separate seed
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.cuda.manual_seed_all(42)

# Some operations on a GPU are implemented stochastic for efficiency
# Ensure that all operations are deterministic on GPU for reproducibility
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

Is the GPU available? True
Device cuda


# Prepare Datasets

In [5]:
ucc_train_filename = '/content/drive/My Drive/Colab Notebooks/AISI_project/data/ucc_train_no_scores.csv'
ucc_test_filename = '/content/drive/My Drive/Colab Notebooks/AISI_project/data/ucc_test_no_scores.csv'
#ucc_train_filename = '/content/drive/My Drive/Colab Notebooks/AISI_project/data/ucc_train_confident_no_scores.csv'
#ucc_test_filename = '/content/drive/My Drive/Colab Notebooks/AISI_project/data/ucc_test_confident_no_sarcasm.csv'

reddit_train_filename = '/content/drive/My Drive/Colab Notebooks/AISI_project/data/reddit_train_balanced.csv'
reddit_test_filename = '/content/drive/My Drive/Colab Notebooks/AISI_project/data/reddit_test_balanced.csv'

In [6]:
max_length = 1000

# Tokenize data

In [7]:
def preprocess_ucc(example):
    '''Method for preprocessing the UCC dataset'''

    # Gather list of positive labels
    all_labels = []
    for class_ in classes:
        if example[class_] == 1:
            all_labels.append(class_)

    # Convert labels to a vector of binary values
    labels = [0.0 for i in range(len(classes))]
    for label in all_labels:
        label_id = class2id[label]
        labels[label_id] = 1

    example = tokenizer(example['comment'], padding='max_length', truncation=True)
    example['labels'] = labels

    return example

def preprocess_reddit(example):

    if example['labels'] == 'abuse':
        example['labels'] = 1.0
    elif example['labels'] == 'non_abuse':
        example['labels'] = 0.0
    else:
        sys.error(1)

    example = tokenizer(example['text'], padding='max_length', truncation=True)
    return example


ucc = load_dataset('csv', data_files={'train': ucc_train_filename, 'test': ucc_test_filename})
reddit = load_dataset('csv', data_files={'train': reddit_train_filename, 'test': reddit_test_filename})

# Dicts for the UCC dataset to convert between class name and class numerical id
classes = [class_ for class_ in list(ucc['train'].features)[1:] if class_]
class2id = {class_:id for id, class_ in enumerate(classes)}
id2class = {id:class_ for class_, id in class2id.items()}
# Dicts for the Reddit dataset to convert between class name and binary label
class2binary = {'abuse': 1, 'non_abuse': 0}
binary2class = {1:'abuse', 0:'non_abuse'}

# Tokenize Data
tokenizer = AutoTokenizer.from_pretrained('microsoft/deberta-v3-small', model_max_length=max_length, use_fast=False)
tokenized_ucc = ucc.map(preprocess_ucc)
tokenized_reddit = reddit.map(preprocess_reddit)

# Delete class columns from reddit dataset now that we represent the labels as vecs
tokenized_ucc = tokenized_ucc.remove_columns(classes)


Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


...

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/2872 [00:00<?, ? examples/s]

Map:   0%|          | 0/4425 [00:00<?, ? examples/s]

Map:   0%|          | 0/2281 [00:00<?, ? examples/s]

Map:   0%|          | 0/410 [00:00<?, ? examples/s]

In [8]:
print(tokenized_ucc, '\n')
print(tokenized_reddit)

DatasetDict({
    train: Dataset({
        features: ['comment', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2872
    })
    test: Dataset({
        features: ['comment', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 4425
    })
}) 

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2281
    })
    test: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 410
    })
})


### Finetuning model in native pytorch:

In [ ]:
# To prepare the tokenized datasets for training in native pytorch, do the following:

# The model does not accept raw text as an input. Remove the 'comment' and 'text' columns:
tokenized_ucc = tokenized_ucc.remove_columns(['comment'])
tokenized_reddit = tokenized_reddit.remove_columns(['text'])

# Set the format of the dataset to return PyTorch tensors instead of lists:
tokenized_ucc.set_format('torch')
tokenized_reddit.set_format('torch')

# Split the tokenized data into train, test
ucc_train_dataset = tokenized_ucc['train'].shuffle(seed=6)
ucc_eval_dataset = tokenized_ucc['test'].shuffle(seed=6)
reddit_train_dataset = tokenized_reddit['train'].shuffle(seed=6)
reddit_eval_dataset = tokenized_reddit['test'].shuffle(seed=6)

# Note: pytorch requires the label column to be named 'labels'. Change if needed.

In [ ]:
print(ucc_train_dataset)
print(ucc_eval_dataset, '\n')
print(reddit_train_dataset)
print(reddit_eval_dataset)

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 2872
})
Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 4425
}) 

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2281
})
Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 410
})


# Experiments

## MTL Architecture: Shared base transformer model and separate task heads

## Initialize LLM

In [ ]:
# Set up the LLM
llm = AutoModel.from_pretrained('microsoft/deberta-v3-small')
llm = llm.to(device)

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

In [ ]:
# Examine output of LLM by running a test batch through
sample = reddit_eval_dataset[0:2]
inputs = {k: v.to(device) for k, v in sample.items() if k != 'labels'}
outputs = llm(**inputs)

print('Object type: ', type(outputs))
print('Output format (shape): ',outputs[0].shape)      # shape is (N,M,S) where N is num samples, M is num words, and S is size of output vec
print('Output used as input for the classifier (shape): ', outputs[0][:,0,:].shape)

Object type:  <class 'transformers.modeling_outputs.BaseModelOutput'>
Output format (shape):  torch.Size([2, 1000, 768])
Output used as input for the classifier (shape):  torch.Size([2, 768])


## Initialize MTL model and train!

In [ ]:
class MTLTextClassification(nn.Module):
    def __init__(self, llm, activation=nn.LeakyReLU()):
        super(MTLTextClassification, self).__init__()

        self.llm = llm
        self.activation = activation

        self.feature_extractor = torch.nn.Sequential(
            nn.Dropout(p=0.2),
            torch.nn.Linear(768, 64),
            torch.nn.LeakyReLU(),
        )

        self.reddit_out = torch.nn.Linear(64, 1)
        self.ucc_out = torch.nn.Linear(64, 5)

    def forward(self, input_ids, token_type_ids, attention_mask, taskid):
        # Get embedding from the LLM we are tuning. Use last hidden state of embedding for classification tasks.
        x = self.llm(input_ids=input_ids, token_type_ids=token_type_ids, attention_mask=attention_mask)
        x = x["last_hidden_state"][:,0,:]

        # Extract shared features for both tasks
        features = self.feature_extractor(x)

        # Branch on task
        if taskid == 1:
            logits = self.ucc_out(features)     # Get output for task 1: UCC classification (BCEWithLogitsLoss)
        elif taskid == 2:
            logits = self.reddit_out(features)  # Get output for task 2: Abuse/Non-abuse classification (BCEWithLogitsLoss)

        return logits


In [ ]:
# Create combined data loaders for both tasks using pytorch-lightning
train_loaders = {
    'a': DataLoader(ucc_train_dataset, shuffle=True, batch_size=4),
    'b': DataLoader(reddit_train_dataset, shuffle=True, batch_size=4),
}
test_loaders = {
    'a': DataLoader(ucc_eval_dataset, shuffle=True, batch_size=4),
    'b': DataLoader(reddit_eval_dataset, shuffle=True, batch_size=4),
}

train_combined_loader = CombinedLoader(train_loaders, 'max_size_cycle')

# Define params of LLM as trainable
for param in llm.parameters():
    param.requires_grad = True

# Initialize Custom MTL Model and optimizer
mtl_model = MTLTextClassification(llm=llm)
mtl_model = mtl_model.to(device)
optimizer = torch.optim.AdamW(mtl_model.parameters(), lr=5e-5)

# Initialize Loss methods for both tasks
ucc_loss = nn.BCEWithLogitsLoss()
reddit_loss = nn.BCEWithLogitsLoss()

# Create the default learning rate scheduler from Trainer:
num_epochs = 3
num_training_steps = num_epochs * len(ucc_train_dataset)
lr_scheduler = get_scheduler(name='linear', optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

In [ ]:
# Examine number of parameters
total_params = sum(p.numel() for p in mtl_model.parameters())
total_params_trainable = sum(p.numel() for p in mtl_model.parameters() if p.requires_grad)
print("Number of parameters: ", total_params)
print("Number of trainable parameters: ", total_params_trainable)

total_params_llm = sum(p.numel() for p in llm.parameters())
total_params_trainable_llm = sum(p.numel() for p in llm.parameters() if p.requires_grad)
print("Number of parameters: ", total_params_llm)
print("Number of trainable parameters: ", total_params_trainable_llm)
print(total_params - total_params_llm)

Number of parameters:  141353926
Number of trainable parameters:  141353926
Number of parameters:  141304320
Number of trainable parameters:  141304320
49606


In [ ]:
def train(mtl_model, optimizer, train_combined_loader, eta=1.0, num_epochs=5):

    mtl_model.train()

    # Record losses per epoch
    loss_per_epoch = []
    reddit_loss_per_epoch = []
    ucc_loss_per_epoch = []

    # Record accuracies per epoch
    total_accs = []
    reddit_accs = []
    ucc_accs = []

    for i in tqdm(range(num_epochs)):

        # Keep track of correct predictions for accuracy reporting.
        ucc_correct = 0
        reddit_correct = 0
        total_correct = 0
        num_preds = 0

        loss_per_batch = []
        reddit_losses = []
        ucc_losses = []

        for batch, batch_idx, dataloader_idx in train_combined_loader:

            # Get data batches for both tasks.
            ucc_batch = batch['a']
            reddit_batch = batch['b']

            # Push batches to device.
            ucc_inputs = {k: v.to(device) for k, v in ucc_batch.items() if k != 'labels'}
            reddit_inputs = {k: v.to(device) for k, v in reddit_batch.items() if k != 'labels'}
            ucc_labels = ucc_batch['labels'].to(device)
            reddit_labels = reddit_batch['labels'].to(device)

            # Get predictions for both tasks for the current batch of data.
            ucc_preds = mtl_model(**ucc_inputs, taskid=1)
            reddit_preds = mtl_model(**reddit_inputs, taskid=2)
            ucc_preds = ucc_preds.squeeze(dim=1)                # output is [Batch_size, 1] but we want [Batch_size]
            reddit_preds = reddit_preds.squeeze(dim=1)

            # Calculate loss.
            task1_loss = ucc_loss(ucc_preds, ucc_labels)
            task2_loss = reddit_loss(reddit_preds, reddit_labels)
            if eta == 1.0:
                loss = task1_loss + task2_loss
            else:
                loss = eta*task1_loss + (1-eta)*task2_loss

            # Record loss per batch
            loss_per_batch.append(loss.item())
            ucc_losses.append(task1_loss.item())
            reddit_losses.append(task2_loss.item())

            # Record accuracy for the current batch.
            num_preds += ucc_labels.shape[0]

            ucc_preds = torch.sigmoid(ucc_preds)                    # Convert logits to probabilities for multi-label classification
            ucc_preds_labels = (ucc_preds >= 0.5).long()            # Binarize predictions to 0 and 1
            ucc_correct += (ucc_preds_labels == ucc_labels).sum()   # Count up number of correct predictions for this batch

            reddit_preds = torch.sigmoid(reddit_preds)              # Repeat for reddit task
            reddit_preds_labels = (reddit_preds >= 0.5).long()
            reddit_correct += (reddit_preds_labels == reddit_labels).sum()

            # Update the model
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            lr_scheduler.step()

            # Release memory on GPU
            del ucc_inputs, reddit_inputs, ucc_labels, reddit_labels
            del ucc_preds, ucc_preds_labels, reddit_preds, reddit_preds_labels, loss
            torch.cuda.empty_cache()

        # Record loss per epoch
        loss_per_epoch.append(loss_per_batch)
        reddit_loss_per_epoch.append(reddit_losses)
        ucc_loss_per_epoch.append(ucc_losses)

        # Calculate and record accuracies.
        ucc_accuracy = ucc_correct / (num_preds*5)
        ucc_accs.append(ucc_accuracy)

        reddit_accuracy = reddit_correct / num_preds
        reddit_accs.append(reddit_accuracy)

        total_accuracy = (ucc_correct + reddit_correct) / (num_preds + (5*num_preds))
        total_accs.append(total_accuracy)

    return loss_per_epoch, reddit_loss_per_epoch, ucc_loss_per_epoch, total_accs, reddit_accs, ucc_accs


In [ ]:
# Run Training on model
loss_per_epoch, reddit_losses, ucc_losses, total_accs, reddit_accs, ucc_accs = train(mtl_model, optimizer, train_combined_loader, eta=1.0, num_epochs=5)

# Save the model
state_dict = mtl_model.state_dict()
model_name = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/mtl_confident.tar'
torch.save(state_dict, model_name)

llm_state_dict = mtl_model.llm.state_dict()
llm_name = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/llm_confident.tar'
torch.save(llm_state_dict, llm_name)

# Save training results
results_dict = {
    'loss_per_epoch': loss_per_epoch, 'reddit_losses': reddit_losses, 'ucc_losses': ucc_losses,
    'total_accs': total_accs, 'reddit_accs': reddit_accs, 'ucc_accs': ucc_accs
}
results_file = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/results_confident.pickle'
with open(results_file, 'wb') as handle:
    pickle.dump(results_dict, handle, protocol=pickle.HIGHEST_PROTOCOL)


  0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
# Load models for Evaluation

llm_state_dict = torch.load(llm_name)
saved_llm = AutoModel.from_pretrained('microsoft/deberta-v3-small')
saved_llm.load_state_dict(llm_state_dict)

state_dict = torch.load(model_name)
saved_mtl_model = MTLTextClassification(llm=saved_llm)
saved_mtl_model.load_state_dict(state_dict)
saved_mtl_model = saved_mtl_model.to(device)

# Verify that the parameters are the same
#print("Original model\n", mtl_model.state_dict()['llm.embeddings.LayerNorm.weight'][0:5])
#print("Loaded model\n", saved_mtl_model.state_dict()['llm.embeddings.LayerNorm.weight'][0:5])
#print("\nOriginal model\n", mtl_model.state_dict()['ucc_out.bias'])
#print("Loaded model\n", saved_mtl_model.state_dict()['ucc_out.bias'])

# Create data loader for testing
test_combined_loader = CombinedLoader(test_loaders, 'max_size_cycle')

In [ ]:
def test(mtl_model, test_combined_loader, eta=1.0):
    mtl_model.eval()

    # Record testing losses per batch
    loss_per_batch = []
    reddit_losses = []
    ucc_losses = []

    # Record accuracies per batch
    total_accs = []
    reddit_accs = []
    ucc_accs = []

    # Keep track of correct predictions for accuracy reporting.
    ucc_correct = 0
    reddit_correct = 0
    total_correct = 0
    num_preds = 0

    with torch.no_grad():
        for batch, batch_idx, dataloader_idx in test_combined_loader:

            # Get data batches for both tasks.
            ucc_batch = batch['a']
            reddit_batch = batch['b']

            # Push batches to device.
            ucc_inputs = {k: v.to(device) for k, v in ucc_batch.items() if k != 'labels'}
            reddit_inputs = {k: v.to(device) for k, v in reddit_batch.items() if k != 'labels'}
            ucc_labels = ucc_batch['labels'].to(device)
            reddit_labels = reddit_batch['labels'].to(device)

            # Get predictions for both tasks for the current batch of data.
            ucc_preds = mtl_model(**ucc_inputs, taskid=1)
            reddit_preds = mtl_model(**reddit_inputs, taskid=2)
            ucc_preds = ucc_preds.squeeze(dim=1)                # output is [Batch_size, 1] but we want [Batch_size]
            reddit_preds = reddit_preds.squeeze(dim=1)

            # Calculate loss.
            task1_loss = ucc_loss(ucc_preds, ucc_labels)
            task2_loss = reddit_loss(reddit_preds, reddit_labels)
            if eta == 1.0:
                loss = task1_loss + task2_loss
            else:
                loss = eta*task1_loss + (1-eta)*task2_loss

            loss_per_batch.append(loss.item())
            ucc_losses.append(task1_loss.item())
            reddit_losses.append(task2_loss.item())

            # Record accuracy for the current batch.
            num_preds += ucc_labels.shape[0]

            ucc_preds = torch.sigmoid(ucc_preds)                    # Convert logits to probabilities for multi-label classification
            ucc_preds_labels = (ucc_preds >= 0.5).long()            # Binarize predictions to 0 and 1
            ucc_correct += (ucc_preds_labels == ucc_labels).sum()   # Count up number of correct predictions for this batch

            reddit_preds = torch.sigmoid(reddit_preds)              # Repeat for reddit task
            reddit_preds_labels = (reddit_preds >= 0.5).long()
            reddit_correct += (reddit_preds_labels == reddit_labels).sum()

            # Release memory on GPU
            del ucc_inputs, reddit_inputs, ucc_labels, reddit_labels
            del ucc_preds, ucc_preds_labels, reddit_preds, reddit_preds_labels, loss
            torch.cuda.empty_cache()

    # Calculate and record accuracies.
    ucc_accuracy = ucc_correct / (num_preds * 5)
    ucc_accs.append(ucc_accuracy)

    reddit_accuracy = reddit_correct / num_preds
    reddit_accs.append(reddit_accuracy)

    total_accuracy = (ucc_correct + reddit_correct) / (num_preds + (5*num_preds))
    total_accs.append(total_accuracy)

    return loss_per_batch, reddit_losses, ucc_losses, total_accs, reddit_accs, ucc_accs

In [ ]:
loss_per_batch, reddit_losses_test, ucc_losses_test, total_accs_test, reddit_accs_test, ucc_accs_test = test(saved_mtl_model, test_combined_loader, eta=1.0)

# Save testing results
test_results_dict = {
    'loss_per_batch': loss_per_batch, 'reddit_losses': reddit_losses_test, 'ucc_losses': ucc_losses_test,
    'total_accs': total_accs_test, 'reddit_accs': reddit_accs_test, 'ucc_accs': ucc_accs_test
}
test_results_file = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/test_results_confident.pickle'
with open(test_results_file, 'wb') as handle:
    pickle.dump(test_results_dict, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
# Clean up
del mtl_model, llm
torch.cuda.empty_cache()

In [ ]:
# Clean up
del saved_mtl_model, saved_llm
torch.cuda.empty_cache()

# Single-Task Learning Baselines



In [ ]:
def stl_training(model, loader, optimizer, loss_fn, taskid):
    model.train()
    losses = []
    accs = []
    for epoch in tqdm(range(5)):
        correct_preds = 0.0
        num_preds = 0.0
        batch_losses = []
        for batch in loader:
            batch_labels = batch['labels']
            batch_labels = batch_labels.to(device)
            batch = {k: v.to(device) for k, v in batch.items() if k != 'labels'}

            outputs = model(**batch, taskid=taskid)
            outputs = outputs.squeeze(dim=1)
            loss = loss_fn(outputs, batch_labels)
            batch_losses.append(loss.item())

            # Record accuracy for the current batch.
            num_preds += batch_labels.shape[0]
            preds = torch.sigmoid(outputs)                      # Convert logits to probabilities for multi-label classification
            preds_labels = (preds >= 0.5).long()                # Binarize predictions to 0 and 1
            correct_preds += (preds_labels == batch_labels).sum()   # Count up number of correct predictions for this batch

            # Update model
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            lr_scheduler.step()

            # Release memory on GPU
            del batch, batch_labels
            del outputs, preds, preds_labels, loss
            torch.cuda.empty_cache()

        losses.append(batch_losses)

        if taskid == 1:
            epoch_acc = correct_preds / (5*num_preds)
        else:
            epoch_acc = correct_preds / (num_preds)
        accs.append(epoch_acc)

    return {'Loss': losses, 'Accuracy': accs}

In [ ]:
def stl_testing(model, loader, loss_fn, taskid):
    model.eval()
    losses = []
    accs = []
    with torch.no_grad():
        correct_preds = 0.0
        num_preds = 0.0
        batch_losses = []
        for batch in loader:
            batch_labels = batch['labels']
            batch_labels = batch_labels.to(device)
            batch = {k: v.to(device) for k, v in batch.items() if k != 'labels'}

            outputs = model(**batch, taskid=taskid)
            outputs = outputs.squeeze(dim=1)
            loss = loss_fn(outputs, batch_labels)
            batch_losses.append(loss.item())

            # Record accuracy for the current batch.
            num_preds += batch_labels.shape[0]
            preds = torch.sigmoid(outputs)                      # Convert logits to probabilities for multi-label classification
            preds_labels = (preds >= 0.5).long()                # Binarize predictions to 0 and 1
            correct_preds += (preds_labels == batch_labels).sum()   # Count up number of correct predictions for this batch

            # Release memory on GPU
            del batch, batch_labels
            del outputs, preds, preds_labels, loss
            torch.cuda.empty_cache()

        losses.append(batch_losses)

        if taskid == 1:
            acc = correct_preds / (5*num_preds)
        else:
            acc = correct_preds / (num_preds)

    return {'Loss': losses, 'Accuracy': acc}

## UCC STL Baseline

In [ ]:
# Train model on UCC dataset using STL only
ucc_llm = AutoModel.from_pretrained('microsoft/deberta-v3-small')
ucc_stl = MTLTextClassification(llm=ucc_llm)
ucc_stl = ucc_stl.to(device)
ucc_optimizer = torch.optim.AdamW(ucc_stl.parameters(), lr=5e-5)
ucc_train_loader = DataLoader(ucc_train_dataset, batch_size=4)
ucc_test_loader = DataLoader(ucc_eval_dataset, batch_size=4)
ucc_loss = nn.BCEWithLogitsLoss()

# Create the default learning rate scheduler from Trainer:
num_epochs = 3
num_training_steps = num_epochs * len(ucc_train_dataset)
lr_scheduler = get_scheduler(name='linear', optimizer=ucc_optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

# Train the UCC STL Model and save the results
train_results = stl_training(ucc_stl, ucc_train_loader, ucc_optimizer, loss_fn=ucc_loss, taskid=1)
results_file = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/ucc_stl_train_results_9_11.pickle'
with open(results_file, 'wb') as f:
    pickle.dump(train_results, f)

# Save a checkpoint of the trained model
state_dict = ucc_stl.state_dict()
model_name = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/ucc_stl_9_11.tar'
torch.save(state_dict, model_name)
llm_state_dict = ucc_stl.llm.state_dict()
llm_name = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/ucc_stl_llm_9_11.tar'
torch.save(llm_state_dict, llm_name)

# Evaluate the UCC STL Model and save the results
test_results = stl_testing(ucc_stl, ucc_test_loader, loss_fn=ucc_loss, taskid=1)
results_file = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/ucc_stl_test_results_9_11.pickle'
with open(results_file, 'wb') as f:
    pickle.dump(test_results, f)

  0%|          | 0/5 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
del ucc_llm, ucc_stl, ucc_optimizer, ucc_train_loader, ucc_test_loader, ucc_loss, lr_scheduler, train_results, test_results
torch.cuda.empty_cache()

## Reddit STL Baseline

In [ ]:
# Train model on Reddit Dataset using STL only
reddit_llm = AutoModel.from_pretrained('microsoft/deberta-v3-small')
reddit_stl = MTLTextClassification(llm=reddit_llm)
reddit_stl = reddit_stl.to(device)
reddit_optimizer = torch.optim.AdamW(reddit_stl.parameters(), lr=5e-5)
reddit_train_loader = DataLoader(reddit_train_dataset, batch_size=4)
reddit_test_loader = DataLoader(reddit_eval_dataset, batch_size=4)
reddit_loss = nn.BCEWithLogitsLoss()

# Create the default learning rate scheduler from Trainer:
num_epochs = 3
num_training_steps = num_epochs * len(reddit_train_dataset)
lr_scheduler = get_scheduler(name='linear', optimizer=reddit_optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

# Train the Reddit STL Model and save the results
train_results = stl_training(reddit_stl, reddit_train_loader, reddit_optimizer, loss_fn=reddit_loss, taskid=2)
results_file = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/reddit_stl_train_results_9_11.pickle'
with open(results_file, 'wb') as f:
    pickle.dump(train_results, f)

# Save a checkpoint of the trained model
state_dict = reddit_stl.state_dict()
model_name = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/reddit_stl_9_11.tar'
torch.save(state_dict, model_name)


  0%|          | 0/5 [00:00<?, ?it/s]

NameError: name 'ucc_stl' is not defined

In [ ]:
llm_state_dict = reddit_stl.llm.state_dict()
llm_name = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/reddit_stl_llm_9_11.tar'
torch.save(llm_state_dict, llm_name)

# Evaluate the UCC STL Model and save the results
test_results = stl_testing(reddit_stl, reddit_test_loader, loss_fn=reddit_loss, taskid=2)
results_file = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/reddit_stl_test_results_9_11.pickle'
with open(results_file, 'wb') as f:
    pickle.dump(test_results, f)

# Examine Results of STL Reddit model

In [ ]:
def load_saved_model(model_file, llm_file, train_results_file, test_results_file):
    # load LLM
    llm_state_dict = torch.load(llm_file)
    saved_llm = AutoModel.from_pretrained('microsoft/deberta-v3-small')
    saved_llm.load_state_dict(llm_state_dict)
    # load model
    state_dict = torch.load(model_file)
    saved_model = MTLTextClassification(llm=saved_llm)
    saved_model.load_state_dict(state_dict)

    # load training and testing results
    with open(train_results_file, 'rb') as handle:
        train_results = pickle.load(handle)
    with open(test_results_file, 'rb') as handle:
        test_results = pickle.load(handle)

    return saved_model, saved_llm, train_results, test_results

In [ ]:
# Load Model
llm_file = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/reddit_stl_llm_9_11.tar'
model_file = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/reddit_stl_9_11.tar'
train_file = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/reddit_stl_train_results_9_11.pickle'
test_file = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/reddit_stl_test_results_9_11.pickle'
stl_model, llm, train_results, test_results = load_saved_model(model_file, llm_file, train_file, test_file)

<ipython-input-6-f6791011a158>:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  llm_state_dict = torch.load(llm_file)
<ipython-input-6-f6791011a158>:7: FutureWarning: You ar

In [ ]:
print(test_results['Loss'])
print('\n', test_results['Accuracy'])

[[0.0017378858756273985, 1.445273995399475, 0.0020258352160453796, 0.005739208776503801, 0.0013323253951966763, 0.0017290764953941107, 0.020238973200321198, 1.1217752695083618, 1.730180263519287, 1.2015475034713745, 0.043765194714069366, 0.006372968666255474, 0.0016844137571752071, 0.35676825046539307, 0.0394015908241272, 0.0016695242375135422, 0.00457778898999095, 0.0013407713267952204, 0.9746675491333008, 0.0013770521618425846, 0.0013438957976177335, 0.001400818582624197, 0.0017278199084103107, 0.00923764705657959, 0.015899600461125374, 0.0038161668926477432, 0.007173679303377867, 0.04238396883010864, 0.0021617815364152193, 0.0016937963664531708, 0.0016554074827581644, 1.325185775756836, 0.0016742297448217869, 0.002033460419625044, 0.0013262293068692088, 0.001988949254155159, 0.0020692963153123856, 0.0018841670826077461, 0.0013550750445574522, 0.0014293561689555645, 0.010895080864429474, 0.0018558991141617298, 0.002037791768088937, 0.002393960952758789, 0.00218567019328475, 0.4866856

In [ ]:
# Create the list of custon inference examples
abuse_ex1 = tokenized_reddit['test'][3]['text']
custom_manipulation_dialogue = "give me your password. what do you have to hide? if you really loved me you'd give it to me"
custom_belittling_ex = "my boyfriend asked me why I posted this one pic. he said that I look really desperate for likes and that the pic is not flattering"
custom_nonabuse = "buying a wedding gift for best friend (guy) who got married, but not his wife. is that ok? one of my best friends got married about a year ago and i couldn't go because i'm out of the country. i didn't send them any gifts and now i want to send a watch to him. i've never met his wife as i've been overseas for about 5 years. would that be bad to send only him a gift? i attempted to start a conversation with her about 3 years ago on facebook just by saying hi and that her bf was my best friend. she never responded i was the best man in my best friends wedding last year. i gave them a giant stack of comic books for the wedding gift. i have a good relationship with her however and she was completely fine with it and even jokes about it. i think it depends on your relationship with both people and how you think they will feel about it. sure, if you want to send your friend a random gift unrelated to the fact that he got married, then go for it."
inference_examples = [abuse_ex1, custom_manipulation_dialogue, custom_belittling_ex, custom_nonabuse]

In [ ]:
# Run inference on custom abuse and non-abuse examples and examine the output:
stl_model.eval()
for example in inference_examples:
    # tokenize example
    tokenized_example = tokenizer(example, return_tensors='pt')

    with torch.no_grad():
        # Run inference on task 2 (Reddit Abuse/Non-abuse classification)
        logits = stl_model(**tokenized_example, taskid=2)
        pred = torch.sigmoid(logits).item()
        if pred >= 0.5:
            print(f'prediction=ABUSE ({pred:.4f}): {example}')
        else:
            print(f'prediction=NON-ABUSE ({pred:.4f}): {example}')   # All examples should output ABUSE except the last 4...
        print('\n')

prediction=ABUSE (0.9990): a friend told me a man pushed and choked her for not hooking up with him. two weeks later she began a relationship with him. how can i help/offer support? hey everyone. i apologize if this is not the right forum for this kind of help, but i am really just looking for any kind of help and opinions at this point to help out a friend i am concerned about. a couple weeks ago a friend called me at 2am, clearly distressed and disclosed that she had been out at a bar, drinking, and met a guy. she thought he was funny and enjoyed being with him, but when she came out of the bathroom at one point, he tried to kiss her and she leaned away. she then told me at this point he pushed her against the wall and choked her for a few seconds. i did my best to console her and she was able to calm down. it is now a few weeks later and she has informed me that he asked her to be his girlfriend and she has agreed. i do not know the details of any dates they have been on since the i

In [ ]:
# Swap the genders of the above examples!
physical_ex = "a friend told me a woman pushed and choked him for not hooking up with her. two weeks later he began a relationship with her. how can i help/offer support? hey everyone. i apologize if this is not the right forum for this kind of help, but i am really just looking for any kind of help and opinions at this point to help out a friend i am concerned about. a couple weeks ago a friend called me at 2am, clearly distressed and disclosed that he had been out at a bar, drinking, and met a girl. he thought she was funny and enjoyed being with her, but when he came out of the bathroom at one point, she tried to kiss him and he leaned away. he then told me at this point she pushed him against the wall and choked him for a few seconds. i did my best to console him and he was able to calm down. it is now a few weeks later and he has informed me that she asked him to be his boyfriend and he has agreed. i do not know the details of any dates they have been on since the initial incident, but he did tell me he talked to her about it, and she said she was drunk, it wasn't like her, and insisted it will never happen again. to him this was okay, and he said that because he was also drinking maybe he misinterpreted it and that he doesnt remember it well. the problem is that he gave me very vivid details just moments after it happened and it seems like something that is very serious and should not be ignored so easily. from what i have heard, behavior like that is rarely just a totally random one time thing, and i am very scared that it will happen again to him. it seems to fit the pattern of physical abuse followed by a 'honeymoon period' where she promised to be nicer and begin a formal relationship. i guess at this point i am just confused and concerned for him. he is a very smart guy and i would never have expected him to be okay with something like this. he also admitted to me that currently he has been looking for a girlfriend and someone to treat him well, so i am worried he is overlooking some important risk factors for the sake of temporary comfort. basically, is there anything i can say or do at this point to help? i am not accusing her of being a serial abuser or anything without knowing more, but pushing and choking are serious red flags to me, and i am terrified if that happened in public over not being physical enough, then what could happen behind closed doors? my initial reaction was admittedly probably not the best as i expressed my concern and confusion very openly and i am not sure he truly heard it and only defended her further. **summary:** guy met girl at bar. the two get along, but when the girl tries to kiss him he looks away. the girl pushes and chokes him. she began acting nice and two weeks later they are boyfriend and girlfriend. i am concerned something will happen again and i am looking for help to effectively express my concern. i'm sorry for the delay. kinda been going thru my own crap. but yes. all you can do is be a friend. there are normally very complicated reasons why someone does this. it's complicated for me. mostly cause my family saw it, i was stubborn and pursued the relationship, and now at my age idk where to go as isolated as i am. i get that at some point you lose all safety nets, but sometimes that is all that saves you. it could take years. lots of heartache. is it better then reading an obit? i would think so. but i haven't had a friend since 8th grade. before boys were involved. so what do i know. thank you for caring. don't judge. be the loving person he craves. after that, people make their own decisions. make it clear you will always love him and will be there if he wants out, but that you can't stand seeing him hurt. stay in touch, so he knows you are safe, but as fucked up as it it, he has to decide to leave. no one can force that. but him knowing he has somewhere to go when ready means he can escape. keep a window open when he closes the door. one day he'll need it"
gender_swapped_ex = "my girlfriend asked me why I posted this one pic. she said that I look really desperate for likes and that the pic is not flattering"
custom_manipulation_dialogue = "give me your password. what do you have to hide? if you really loved me you'd give it to me"
custom_nonabuse = "buying a wedding gift for best friend (girl) who got married, but not her husband. is that ok? one of my best friends got married about a year ago and i couldn't go because i'm out of the country. i didn't send them any gifts and now i want to send a watch to her. i've never met her husband as i've been overseas for about 5 years. would that be bad to send only her a gift? i attempted to start a conversation with him about 3 years ago on facebook just by saying hi and that his gf was my best friend. he never responded i was the best man in my best friends wedding last year. i gave them a giant stack of comic books for the wedding gift. i have a good relationship with him however and he was completely fine with it and even jokes about it. i think it depends on your relationship with both people and how you think they will feel about it. sure, if you want to send your friend a random gift unrelated to the fact that she got married, then go for it."
gender_swapped_inference_examples = [physical_ex, custom_manipulation_dialogue, gender_swapped_ex, custom_nonabuse]

In [ ]:
# Run inference on custom abuse and non-abuse examples and examine the output:
stl_model.eval()
for example in gender_swapped_inference_examples:
    # tokenize example
    tokenized_example = tokenizer(example, return_tensors='pt')

    with torch.no_grad():
        # Run inference on task 2 (Reddit Abuse/Non-abuse classification)
        logits = stl_model(**tokenized_example, taskid=2)
        pred = torch.sigmoid(logits).item()
        if pred >= 0.5:
            print(f'prediction=ABUSE ({pred:.4f}): {example}')
        else:
            print(f'prediction=NON-ABUSE ({pred:.4f}): {example}')   # All examples should output ABUSE except the last 4...
        print('\n')

prediction=ABUSE (0.9990): a friend told me a woman pushed and choked him for not hooking up with her. two weeks later he began a relationship with her. how can i help/offer support? hey everyone. i apologize if this is not the right forum for this kind of help, but i am really just looking for any kind of help and opinions at this point to help out a friend i am concerned about. a couple weeks ago a friend called me at 2am, clearly distressed and disclosed that he had been out at a bar, drinking, and met a girl. he thought she was funny and enjoyed being with her, but when he came out of the bathroom at one point, she tried to kiss him and he leaned away. he then told me at this point she pushed him against the wall and choked him for a few seconds. i did my best to console him and he was able to calm down. it is now a few weeks later and he has informed me that she asked him to be his boyfriend and he has agreed. i do not know the details of any dates they have been on since the init

# Test MTL Models on Custom Inference Examples

## MTL Default

In [ ]:
# Load Model
llm_file = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/llm_Apr29_01_05_21.tar'
model_file = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/mtl_Apr29_01_05_21.tar'
train_file = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/results_Apr29_01_05_21.pickle'
test_file = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/test_results_Apr29_01_05_21.pickle'
mtl_model, llm, train_results, test_results = load_saved_model(model_file, llm_file, train_file, test_file)

<ipython-input-6-f6791011a158>:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  llm_state_dict = torch.load(llm_file)
<ipython-input-6-f6791011a158>:7: FutureWarning: You ar

In [ ]:
# Run inference on custom abuse and non-abuse examples and examine the output:
mtl_model.eval()
for example in inference_examples:
    # tokenize example
    tokenized_example = tokenizer(example, return_tensors='pt')

    with torch.no_grad():
        # Run inference on task 2 (Reddit Abuse/Non-abuse classification)
        logits = mtl_model(**tokenized_example, taskid=2)
        pred = torch.sigmoid(logits).item()
        if pred >= 0.5:
            print(f'prediction=ABUSE ({pred:.4f}): {example}')
        else:
            print(f'prediction=NON-ABUSE ({pred:.4f}): {example}')   # All examples should output ABUSE except the last 4...
        print('\n')

prediction=ABUSE (0.9993): a friend told me a man pushed and choked her for not hooking up with him. two weeks later she began a relationship with him. how can i help/offer support? hey everyone. i apologize if this is not the right forum for this kind of help, but i am really just looking for any kind of help and opinions at this point to help out a friend i am concerned about. a couple weeks ago a friend called me at 2am, clearly distressed and disclosed that she had been out at a bar, drinking, and met a guy. she thought he was funny and enjoyed being with him, but when she came out of the bathroom at one point, he tried to kiss her and she leaned away. she then told me at this point he pushed her against the wall and choked her for a few seconds. i did my best to console her and she was able to calm down. it is now a few weeks later and she has informed me that he asked her to be his girlfriend and she has agreed. i do not know the details of any dates they have been on since the i

In [ ]:
# Run inference on custom abuse and non-abuse examples and examine the output:
mtl_model.eval()
for example in gender_swapped_inference_examples:
    # tokenize example
    tokenized_example = tokenizer(example, return_tensors='pt')

    with torch.no_grad():
        # Run inference on task 2 (Reddit Abuse/Non-abuse classification)
        logits = mtl_model(**tokenized_example, taskid=2)
        pred = torch.sigmoid(logits).item()
        if pred >= 0.5:
            print(f'prediction=ABUSE ({pred:.4f}): {example}')
        else:
            print(f'prediction=NON-ABUSE ({pred:.4f}): {example}')   # All examples should output ABUSE except the last 4...
        print('\n')

prediction=ABUSE (0.9993): a friend told me a woman pushed and choked him for not hooking up with her. two weeks later he began a relationship with her. how can i help/offer support? hey everyone. i apologize if this is not the right forum for this kind of help, but i am really just looking for any kind of help and opinions at this point to help out a friend i am concerned about. a couple weeks ago a friend called me at 2am, clearly distressed and disclosed that he had been out at a bar, drinking, and met a girl. he thought she was funny and enjoyed being with her, but when he came out of the bathroom at one point, she tried to kiss him and he leaned away. he then told me at this point she pushed him against the wall and choked him for a few seconds. i did my best to console him and he was able to calm down. it is now a few weeks later and he has informed me that she asked him to be his boyfriend and he has agreed. i do not know the details of any dates they have been on since the init

## MTL tau=0.4

In [ ]:
llm_file = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/llm_eta_40-60.tar'
model_file = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/mtl_eta_40-60'
train_file = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/results_eta_40_60.pickle'
test_file = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/test_results_eta_40_60.pickle'
mtl_model, llm, train_results, test_results = load_saved_model(model_file, llm_file, train_file, test_file)

<ipython-input-6-f6791011a158>:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  llm_state_dict = torch.load(llm_file)
<ipython-input-6-f6791011a158>:7: FutureWarning: You ar

In [ ]:
# Run inference on custom abuse and non-abuse examples and examine the output:
mtl_model.eval()
for example in inference_examples:
    # tokenize example
    tokenized_example = tokenizer(example, return_tensors='pt')

    with torch.no_grad():
        # Run inference on task 2 (Reddit Abuse/Non-abuse classification)
        logits = mtl_model(**tokenized_example, taskid=2)
        pred = torch.sigmoid(logits).item()
        if pred >= 0.5:
            print(f'prediction=ABUSE ({pred:.4f}): {example}')
        else:
            print(f'prediction=NON-ABUSE ({pred:.4f}): {example}')   # All examples should output ABUSE except the last 4...
        print('\n')

prediction=ABUSE (0.9971): a friend told me a man pushed and choked her for not hooking up with him. two weeks later she began a relationship with him. how can i help/offer support? hey everyone. i apologize if this is not the right forum for this kind of help, but i am really just looking for any kind of help and opinions at this point to help out a friend i am concerned about. a couple weeks ago a friend called me at 2am, clearly distressed and disclosed that she had been out at a bar, drinking, and met a guy. she thought he was funny and enjoyed being with him, but when she came out of the bathroom at one point, he tried to kiss her and she leaned away. she then told me at this point he pushed her against the wall and choked her for a few seconds. i did my best to console her and she was able to calm down. it is now a few weeks later and she has informed me that he asked her to be his girlfriend and she has agreed. i do not know the details of any dates they have been on since the i

In [ ]:
# Run inference on custom abuse and non-abuse examples and examine the output:
mtl_model.eval()
for example in gender_swapped_inference_examples:
    # tokenize example
    tokenized_example = tokenizer(example, return_tensors='pt')

    with torch.no_grad():
        # Run inference on task 2 (Reddit Abuse/Non-abuse classification)
        logits = mtl_model(**tokenized_example, taskid=2)
        pred = torch.sigmoid(logits).item()
        if pred >= 0.5:
            print(f'prediction=ABUSE ({pred:.4f}): {example}')
        else:
            print(f'prediction=NON-ABUSE ({pred:.4f}): {example}')   # All examples should output ABUSE except the last 4...
        print('\n')

prediction=ABUSE (0.9971): a friend told me a woman pushed and choked him for not hooking up with her. two weeks later he began a relationship with her. how can i help/offer support? hey everyone. i apologize if this is not the right forum for this kind of help, but i am really just looking for any kind of help and opinions at this point to help out a friend i am concerned about. a couple weeks ago a friend called me at 2am, clearly distressed and disclosed that he had been out at a bar, drinking, and met a girl. he thought she was funny and enjoyed being with her, but when he came out of the bathroom at one point, she tried to kiss him and he leaned away. he then told me at this point she pushed him against the wall and choked him for a few seconds. i did my best to console him and he was able to calm down. it is now a few weeks later and he has informed me that she asked him to be his boyfriend and he has agreed. i do not know the details of any dates they have been on since the init

## MTL tau=0.6

In [ ]:
# Load Model
llm_file = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/llm_eta_60-40.tar'
model_file = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/mtl_eta_60-40'
train_file = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/results_eta_60_40.pickle'
test_file = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/test_results_eta_60_40.pickle'
mtl_model, llm, train_results, test_results = load_saved_model(model_file, llm_file, train_file, test_file)

<ipython-input-6-f6791011a158>:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  llm_state_dict = torch.load(llm_file)
<ipython-input-6-f6791011a158>:7: FutureWarning: You ar

In [ ]:
# Run inference on custom abuse and non-abuse examples and examine the output:
mtl_model.eval()
for example in inference_examples:
    # tokenize example
    tokenized_example = tokenizer(example, return_tensors='pt')

    with torch.no_grad():
        # Run inference on task 2 (Reddit Abuse/Non-abuse classification)
        logits = mtl_model(**tokenized_example, taskid=2)
        pred = torch.sigmoid(logits).item()
        if pred >= 0.5:
            print(f'prediction=ABUSE ({pred:.4f}): {example}')
        else:
            print(f'prediction=NON-ABUSE ({pred:.4f}): {example}')   # All examples should output ABUSE except the last 4...
        print('\n')

prediction=ABUSE (0.9954): a friend told me a man pushed and choked her for not hooking up with him. two weeks later she began a relationship with him. how can i help/offer support? hey everyone. i apologize if this is not the right forum for this kind of help, but i am really just looking for any kind of help and opinions at this point to help out a friend i am concerned about. a couple weeks ago a friend called me at 2am, clearly distressed and disclosed that she had been out at a bar, drinking, and met a guy. she thought he was funny and enjoyed being with him, but when she came out of the bathroom at one point, he tried to kiss her and she leaned away. she then told me at this point he pushed her against the wall and choked her for a few seconds. i did my best to console her and she was able to calm down. it is now a few weeks later and she has informed me that he asked her to be his girlfriend and she has agreed. i do not know the details of any dates they have been on since the i

In [ ]:
# Run inference on custom abuse and non-abuse examples and examine the output:
mtl_model.eval()
for example in gender_swapped_inference_examples:
    # tokenize example
    tokenized_example = tokenizer(example, return_tensors='pt')

    with torch.no_grad():
        # Run inference on task 2 (Reddit Abuse/Non-abuse classification)
        logits = mtl_model(**tokenized_example, taskid=2)
        pred = torch.sigmoid(logits).item()
        if pred >= 0.5:
            print(f'prediction=ABUSE ({pred:.4f}): {example}')
        else:
            print(f'prediction=NON-ABUSE ({pred:.4f}): {example}')   # All examples should output ABUSE except the last 4...
        print('\n')

prediction=ABUSE (0.9954): a friend told me a woman pushed and choked him for not hooking up with her. two weeks later he began a relationship with her. how can i help/offer support? hey everyone. i apologize if this is not the right forum for this kind of help, but i am really just looking for any kind of help and opinions at this point to help out a friend i am concerned about. a couple weeks ago a friend called me at 2am, clearly distressed and disclosed that he had been out at a bar, drinking, and met a girl. he thought she was funny and enjoyed being with her, but when he came out of the bathroom at one point, she tried to kiss him and he leaned away. he then told me at this point she pushed him against the wall and choked him for a few seconds. i did my best to console him and he was able to calm down. it is now a few weeks later and he has informed me that she asked him to be his boyfriend and he has agreed. i do not know the details of any dates they have been on since the init

## MTL Confident

In [ ]:
# Load Model
llm_file = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/llm_confident.tar'         # default eta and confident ucc dataset
model_file = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/mtl_confident.tar'
train_file = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/results_confident.pickle'
test_file = '/content/drive/My Drive/Colab Notebooks/AISI_project/checkpoints/test_results_confident.pickle'
mtl_model, llm, train_results, test_results = load_saved_model(model_file, llm_file, train_file, test_file)

<ipython-input-6-f6791011a158>:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  llm_state_dict = torch.load(llm_file)
<ipython-input-6-f6791011a158>:7: FutureWarning: You ar

In [ ]:
# Run inference on custom abuse and non-abuse examples and examine the output:
mtl_model.eval()
for example in inference_examples:
    # tokenize example
    tokenized_example = tokenizer(example, return_tensors='pt')

    with torch.no_grad():
        # Run inference on task 2 (Reddit Abuse/Non-abuse classification)
        logits = mtl_model(**tokenized_example, taskid=2)
        pred = torch.sigmoid(logits).item()
        if pred >= 0.5:
            print(f'prediction=ABUSE ({pred:.4f}): {example}')
        else:
            print(f'prediction=NON-ABUSE ({pred:.4f}): {example}')   # All examples should output ABUSE except the last 4...
        print('\n')

prediction=ABUSE (0.9971): a friend told me a man pushed and choked her for not hooking up with him. two weeks later she began a relationship with him. how can i help/offer support? hey everyone. i apologize if this is not the right forum for this kind of help, but i am really just looking for any kind of help and opinions at this point to help out a friend i am concerned about. a couple weeks ago a friend called me at 2am, clearly distressed and disclosed that she had been out at a bar, drinking, and met a guy. she thought he was funny and enjoyed being with him, but when she came out of the bathroom at one point, he tried to kiss her and she leaned away. she then told me at this point he pushed her against the wall and choked her for a few seconds. i did my best to console her and she was able to calm down. it is now a few weeks later and she has informed me that he asked her to be his girlfriend and she has agreed. i do not know the details of any dates they have been on since the i

In [ ]:
# Run inference on custom abuse and non-abuse examples and examine the output:
mtl_model.eval()
for example in gender_swapped_inference_examples:
    # tokenize example
    tokenized_example = tokenizer(example, return_tensors='pt')

    with torch.no_grad():
        # Run inference on task 2 (Reddit Abuse/Non-abuse classification)
        logits = mtl_model(**tokenized_example, taskid=2)
        pred = torch.sigmoid(logits).item()
        if pred >= 0.5:
            print(f'prediction=ABUSE ({pred:.4f}): {example}')
        else:
            print(f'prediction=NON-ABUSE ({pred:.4f}): {example}')   # All examples should output ABUSE except the last 4...
        print('\n')

prediction=ABUSE (0.9971): a friend told me a woman pushed and choked him for not hooking up with her. two weeks later he began a relationship with her. how can i help/offer support? hey everyone. i apologize if this is not the right forum for this kind of help, but i am really just looking for any kind of help and opinions at this point to help out a friend i am concerned about. a couple weeks ago a friend called me at 2am, clearly distressed and disclosed that he had been out at a bar, drinking, and met a girl. he thought she was funny and enjoyed being with her, but when he came out of the bathroom at one point, she tried to kiss him and he leaned away. he then told me at this point she pushed him against the wall and choked him for a few seconds. i did my best to console him and he was able to calm down. it is now a few weeks later and he has informed me that she asked him to be his boyfriend and he has agreed. i do not know the details of any dates they have been on since the init

# Cleanup when done

In [ ]:
del reddit_llm, reddit_stl, reddit_optimizer, reddit_train_loader, reddit_test_loader, reddit_loss, lr_scheduler, train_results, test_results
torch.cuda.empty_cache()

# Demonstrating Bias

In [ ]:
# Load MTL model
# Run inference on custom example
# Save the probability of abuse that was output
# Run inference on same custom example but with gender swapped
# Save the probability of abuse that was output
# Repeat for STL Task 2 model and other MTL models, if results are similar then just say that in paper without a table of all results

# Graphs
might need to generate new graphs depending on results of training above